In [1]:
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
import math
import random
import seaborn as sns
import matplotlib.pyplot as plt
# Sirve para medir la velocidad de una función
import time
%load_ext line_profiler
import sys
sys.path.insert(1, '/Users/damian/Documents/Tesis maestria/Bootstraps_codigos_bases/Codigos/Funciones_utiles_f')
import funciones_aux_bootstrap_f_r as fab
from scipy.stats import variation

Este método de bootstrap (bayesiano) por votos consiste en lo siguiente

1. Se muestrea $n$ casillas.
2. De las $n$ casillas se muestrean $k$ votos para cada una de las $(1000)$ poblaciones distintas. Es decir, se obtiene un total de $1000$ muestras distintas de votos.
3. Para cada población y cada una de las $1000$ muestras de votos de tamaño $k$ se calculan los pesos Dirichlet, por lo tanto, se calculan en total $k*1000$ distintos pesos Dirichlet.
4. Se completan $1000$ poblaciones.
5. Con las poblaciones se obtienen la estimación puntual.

Se repite este proceso para $l$ bootstraps (normalmente son 1000) y con estos bootstraps se obtienen todas las métricas (errores, tamaño de los intervalos, coberturas, etc).

In [2]:
# Importamos los datos para el análisis
df_act_yuc_i=pd.read_csv(r"/Users/damian/Documents/Tesis maestria/Bootstraps_codigos_bases/Bases/Base elecciones analsis/base_elecciones_yuc_analsis.csv", index_col=0)
df_act_yuc_i

,ID ESTADO,NOMBRE ESTADO,DISTRITO,SECCION,CASILLA,ID CASILLA,ESTATUS ACTA,PAN,PRI,PRD,...,CANDIDATOS NO REGISTRADOS,VOTOS NULOS,LISTA NOMINAL,TOTAL,PARTICIPACIÓN,JOAQUIN_DIAZ_MENA,RENAN_BARRERA_CONCHA,VIDA_ARAVARI_GOMEZ_HERRERA,YAMIL_JASMIN_LOPEZ_MANRIQUE,VOTOS_NULOS_CAND_NO_REGIS
0,31,YUCATÁN,12,1,'0001B1,1,COMPUTADA,171.0,107.0,3.0,...,0.0,12.0,596.0,532.0,0.892617,209.0,290.0,18.0,3.0,12.0
1,31,YUCATÁN,12,1,'0001C1,2,COMPUTADA,177.0,104.0,0.0,...,0.0,11.0,596.0,547.0,0.917785,208.0,299.0,29.0,0.0,11.0
2,31,YUCATÁN,12,1,'0001C2,3,COMPUTADA,176.0,100.0,1.0,...,0.0,13.0,595.0,542.0,0.910924,228.0,289.0,11.0,1.0,13.0
3,31,YUCATÁN,12,2,'0002B1,4,COMPUTADA,46.0,33.0,2.0,...,0.0,3.0,174.0,162.0,0.931034,71.0,79.0,7.0,2.0,3.0
4,31,YUCATÁN,12,4,'0004B1,6,COMPUTADA,110.0,96.0,7.0,...,0.0,14.0,550.0,513.0,0.932727,275.0,207.0,10.0,7.0,14.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2836,31,YUCATÁN,3,1195,'1195V1,2967,COMPUTADA,2.0,0.0,0.0,...,0.0,0.0,0.0,2.0,0.000000,0.0,2.0,0.0,0.0,0.0
2837,31,YUCATÁN,5,1197,'1197V1,2969,COMPUTADA,2.0,1.0,0.0,...,0.0,0.0,0.0,5.0,0.000000,2.0,3.0,0.0,0.0,0.0
2838,31,YUCATÁN,8,1199,'1199V1,2971,COMPUTADA,2.0,0.0,0.0,...,0.0,0.0,0.0,3.0,0.000000,1.0,2.0,0.0,0.0,0.0
2839,31,YUCATÁN,9,1200,'1200V1,2972,COMPUTADA,0.0,0.0,0.0,...,0.0,0.0,0.0,1.0,0.000000,1.0,0.0,0.0,0.0,0.0


In [3]:
# Datos reales de la proporción
prop_reales=df_act_yuc_i[["JOAQUIN_DIAZ_MENA","RENAN_BARRERA_CONCHA","VIDA_ARAVARI_GOMEZ_HERRERA","YAMIL_JASMIN_LOPEZ_MANRIQUE","VOTOS_NULOS_CAND_NO_REGIS"]].sum()/df_act_yuc_i[["JOAQUIN_DIAZ_MENA","RENAN_BARRERA_CONCHA","VIDA_ARAVARI_GOMEZ_HERRERA","YAMIL_JASMIN_LOPEZ_MANRIQUE","VOTOS_NULOS_CAND_NO_REGIS"]].sum().sum()
prop_reales

JOAQUIN_DIAZ_MENA              0.515125
RENAN_BARRERA_CONCHA           0.421366
VIDA_ARAVARI_GOMEZ_HERRERA     0.036792
YAMIL_JASMIN_LOPEZ_MANRIQUE    0.005249
VOTOS_NULOS_CAND_NO_REGIS      0.021467
dtype: float64

In [4]:
num_boot=1000
n_sample=100
n_vot_sub=20000
semilla=3
est_punt_list=[]

inter_list=[]

# Número total de votos
n_votos=int(df_act_yuc_i[["JOAQUIN_DIAZ_MENA","RENAN_BARRERA_CONCHA","VIDA_ARAVARI_GOMEZ_HERRERA","YAMIL_JASMIN_LOPEZ_MANRIQUE","VOTOS_NULOS_CAND_NO_REGIS"]].sum().sum())

# Semilla para las muestras de la Dirichlet
rng = np.random.default_rng()

# Semilla para los conteos de votos (eventualmente se tiene que mover adentro del for para que sea reproducible el código)
rng_2 = np.random.default_rng()

# Votos por casilla
casilla_mue=[]
coeef_porcentajes=[]
coeff_votos=[]
coeff_pesos_boot=[]
for i in range(num_boot):

    # Semilla para que el código sea reproducible
    # Cambiamos el valor en el for
    rng = np.random.default_rng((num_boot*semilla)+i+1)

    df_stra_sam = df_act_yuc_i.sample(n=n_sample, random_state= rng)

    df_stra_sam=np.array(df_stra_sam[["JOAQUIN_DIAZ_MENA","RENAN_BARRERA_CONCHA","VIDA_ARAVARI_GOMEZ_HERRERA","YAMIL_JASMIN_LOPEZ_MANRIQUE","VOTOS_NULOS_CAND_NO_REGIS"]])
    casilla_mue.append(df_stra_sam.sum(0)/df_stra_sam.sum(0).sum())
    
    # Semilla para las muestras de la Dirichlet
    rng = np.random.default_rng((i+1)+(num_boot*(semilla+1)))

    # Submuestra de votos (en este caso son 2000)
    # Corrección del código inicial lo correcto es utilizar la distribución hipergeométrica multivariada.
    # Puesto que, se esta haciendo muestreo sin reemplazo y si utilizas la multinomial es muestreo con reemplazo.
    # La diferencia es mínima pero la hipergeométrica multivariada es la correcta.
    # Sacamos 1000 muestras distintas
    votos_subm=rng.multivariate_hypergeometric(colors=df_stra_sam.sum(0).astype(int), nsample=n_vot_sub, size=1000)
    coeff_votos.append(votos_subm.std(axis=0)/votos_subm.mean(axis=0))

    # Los pesos de la distribución Dirichlet para las 1000 muestras distintas
    pesos_bootstrap=fab.dirichlet_sample(votos_subm, semilla=(i+1)+(num_boot*(semilla+2)))
    coeff_pesos_boot.append(pesos_bootstrap.std(axis=0)/pesos_bootstrap.mean(axis=0))

    # Semilla para las muestras de la Dirichlet
    rng = np.random.default_rng((i+1)+(num_boot*(semilla+3)))

    # Simulamos las 1000 poblaciones de votos
    votos_sim=rng.multinomial(n=n_votos,  pvals=pesos_bootstrap)

    # Obtenemos las proporciones de los votos
    array_res=(votos_sim)/n_votos
    coeef_porcentajes.append(array_res.std(axis=0)/array_res.mean(axis=0))

    # Los valores de los intervalos de probabilidad lower y upper
    inter_prob=np.concatenate((np.apply_along_axis(fab.inter_prob_l, axis=0, arr=array_res),np.apply_along_axis(fab.inter_prob_u, axis=0, arr=array_res)))
    
    # Estimaciones puntuales
    est_puntual=array_res.mean(0)

    # Guardamos las estimaciones puntuales
    est_punt_list.append(est_puntual)

    # Guardamos los intervalos
    inter_list.append(inter_prob)
    
print(df_stra_sam.shape[0])

# Todas las estimaciones puntuales
array_est_punt=np.array(est_punt_list)

array_inter_prob=np.array(inter_list)

100


## Calculos de los coeficientes de variación

In [5]:
np.array(casilla_mue).std(axis=0)/np.array(casilla_mue).mean(axis=0)

array([0.02728585, 0.03553224, 0.10616174, 0.24082485, 0.07875862])

In [6]:
abs(np.array(casilla_mue)-np.array(prop_reales.T)).std(0)/abs(np.array(casilla_mue)-np.array(prop_reales.T)).mean(0)


array([0.74202077, 0.72361831, 0.74219206, 0.76255337, 0.96757874])

In [14]:
abs(np.array(casilla_mue)-np.array(prop_reales.T)).mean(0)

array([0.0112856 , 0.01216279, 0.00313227, 0.0010037 , 0.00121647])

In [7]:
np.mean(np.array(coeff_votos),axis=0)

array([0.00501906, 0.00604788, 0.02657369, 0.07248769, 0.03491302])

In [8]:
np.mean(np.array(coeff_pesos_boot),axis=0)

array([0.00850245, 0.0102479 , 0.04506669, 0.12301806, 0.0591602 ])

In [9]:
np.mean(np.array(coeef_porcentajes),axis=0)

array([0.0085474 , 0.01030335, 0.04530274, 0.12366383, 0.05947743])

In [10]:
a,b,c=fab.metricas_bootstrap(prop_reales, array_inter_prob, array_est_punt, porcenta_tama=2000)

In [11]:
c

,Candidato,Numero_total_cob,Cobertura,Porcenta_tama
0,JOAQUIN_DIAZ_MENA,439,0.439,2000
1,RENAN_BARRERA_CONCHA,407,0.407,2000
2,VIDA_ARAVARI_GOMEZ_HERRERA,580,0.580,2000
3,VOTOS_NULOS_CAND_NO_REGIS,897,0.897,2000
4,YAMIL_JASMIN_LOPEZ_MANRIQUE,680,0.680,2000


In [12]:
a.groupby('Candidato').agg({'Error':'std'})/a.groupby('Candidato').agg({'Error':'mean'})

,Error
Candidato,
JOAQUIN_DIAZ_MENA,0.742006
RENAN_BARRERA_CONCHA,0.724201
VIDA_ARAVARI_GOMEZ_HERRERA,0.741160
VOTOS_NULOS_CAND_NO_REGIS,0.969146
YAMIL_JASMIN_LOPEZ_MANRIQUE,0.762244


In [13]:
a.groupby('Candidato').agg({'Longitud_intervalo':'std'})/a.groupby('Candidato').agg({'Longitud_intervalo':'mean'})

,Longitud_intervalo
Candidato,
JOAQUIN_DIAZ_MENA,0.029627
RENAN_BARRERA_CONCHA,0.030060
VIDA_ARAVARI_GOMEZ_HERRERA,0.057304
VOTOS_NULOS_CAND_NO_REGIS,0.048101
YAMIL_JASMIN_LOPEZ_MANRIQUE,0.119023


# Conclusiones

- Como se observa en las coberturas de las casillas, los votos nulos (y candidatos no registrados) tienen coberturas muy altas aún cuando solo 100 casillas (alrededor de 0.87), esto, comparado con los otros dos candidatos con menos del 5% de votos (Yamil Jasmín López Manrique y Vida Aravari Gómez Herrera). Por otra parte, a diferencia de los candidatos que también tienen coberturas altas (Joaquín Díaz Mena y Renan Barrera Concha) estos tienen porcentajes de votos altos y por ende al momento de hacer la muestra de votos estás salen más altas. Esto último causa que si los votos son muy grandes ($n \geq 1000$) la varianza de su entrada en distribución Dirichlet disminuya y se pareza mucho a hacer bootstrap frecuentista. Por lo que, las submuestras de votos se parezcan a las de casillas. Mientras que, para los candidatos con porcentaje bajo tienen pocos votos en la submuestra y por ende mayor varianza en sus entradas Dirichlet. Por todas las razones anteriores, se explica porque los votos nulos (y candidatos no registrados) son casi siempre el candidato con la cobertura más alta en los métodos 1 y 2 (muestrear casillas y luego votos). Esto se observa más en el método 2.